# The Sun's Supergranulation — Implementation / 구현

**Paper**: Rieutord, M. & Rincon, F., "The Sun's Supergranulation", *Living Rev. Solar Phys.*, **7**, 2 (2010). DOI: [10.12942/lrsp-2010-2](https://doi.org/10.12942/lrsp-2010-2)

## Goal / 목표

Since this is a review paper, this notebook reproduces the **key observational and theoretical concepts** rather than a single algorithm. We construct synthetic supergranulation flows and analyse them with the diagnostics used throughout the review.

리뷰 논문이므로 단일 알고리즘이 아닌 **핵심 관측·이론 개념**을 재현합니다. 합성 초과립 유동을 만들고 리뷰 전반에서 사용된 진단 기법으로 분석합니다.

**Sections / 섹션**

1. Solar dissipation hierarchy — verify $\ell_\nu \ll \ell_\eta \ll \ell_\kappa \ll L_G \ll L_{\rm SG}$ / 산란 척도 위계 확인
2. Synthetic supergranulation velocity field on a 2D Cartesian box / 2D 직교 격자 합성 SG 유동 생성
3. Horizontal kinetic energy spectrum $E_h(k)$ — reproduce the $\sim 36$ Mm peak (Fig. 4) / 수평 운동에너지 스펙트럼 재현
4. Anticyclonic signature — divergence-vorticity correlation vs latitude (Fig. 5) / 반사이클론 시그너처
5. Magnetic network formation by passive advection of flux to SG boundaries (Fig. 7) / 자기 네트워크 형성
6. Kinetic vs magnetic energy crossover — equipartition argument (Eqs. 7–8, Fig. 15) / 등분배 논의
7. Summary connecting to Paper #16 (Nordlund 2009) and beyond / 요약

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

rng = np.random.default_rng(seed=42)

## Part 1: Solar dissipation hierarchy / 태양 산란 척도 위계

Reproduce the scale ordering from §2.2 of the review.

$$
\ell_\nu \sim Re^{-3/4} L, \quad \ell_\eta \sim Pm^{-3/4}\ell_\nu, \quad \ell_\kappa \sim Pr^{-3/4}\ell_\nu
$$

Using the Sun's photospheric values: $\nu \sim 10^{-3}\,\mathrm{m^2/s}$, $L \sim L_G \sim 1$ Mm, $V \sim 1$ km/s, $Pm \sim 10^{-5}$, $Pr \sim 10^{-7}$.

광구 값을 사용하여 산란 척도 위계 확인.

In [ ]:
def dissipation_scales(L: float, V: float, nu: float, Pm: float, Pr: float) -> dict:
    """Compute Kolmogorov-style dissipation scales.

    Args:
        L: Injection length scale [m].
        V: Injection velocity scale [m/s].
        nu: Kinematic viscosity [m^2/s].
        Pm: Magnetic Prandtl number = nu / eta.
        Pr: Thermal Prandtl number = nu / kappa.

    Returns:
        Dict with viscous, magnetic, and thermal dissipation scales [m].
    """
    Re = L * V / nu
    ell_nu = Re ** (-0.75) * L
    ell_eta = Pm ** (-0.75) * ell_nu
    ell_kappa = Pr ** (-0.75) * ell_nu
    return {"Re": Re, "ell_nu": ell_nu, "ell_eta": ell_eta, "ell_kappa": ell_kappa}


scales = dissipation_scales(L=1e6, V=1e3, nu=1e-3, Pm=1e-5, Pr=1e-7)
L_G = 1e6
L_SG = 30e6

print(f"Reynolds number Re        = {scales['Re']:.2e}")
print(f"Viscous   scale ell_nu    = {scales['ell_nu']*1e3:.3f} mm")
print(f"Magnetic  scale ell_eta   = {scales['ell_eta']:.1f} m")
print(f"Thermal   scale ell_kappa = {scales['ell_kappa']:.1f} m")
print(f"Granulation L_G           = {L_G/1e6:.2f} Mm")
print(f"Supergranulation L_SG     = {L_SG/1e6:.1f} Mm")
print("\nOrdering check / 위계 확인:")
print(f"ell_nu < ell_eta < ell_kappa < L_G < L_SG : "
      f"{scales['ell_nu'] < scales['ell_eta'] < scales['ell_kappa'] < L_G < L_SG}")

In [ ]:
labels = [r'$\ell_\nu$', r'$\ell_\eta$', r'$\ell_\kappa$', r'$L_G$', r'$L_{\rm SG}$']
values = [scales['ell_nu'], scales['ell_eta'], scales['ell_kappa'], L_G, L_SG]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.set_xscale('log')
ax.set_xlim(1e-4, 1e8)
for label, v in zip(labels, values):
    ax.axvline(v, color='C0', lw=2, alpha=0.7)
    ax.text(v, 0.65, label, ha='center', fontsize=14)
    ax.text(v, 0.35, f'{v:.1e} m', ha='center', fontsize=9, color='gray', rotation=90)
ax.set_yticks([])
ax.set_xlabel('Scale [m] (log) — 8 orders of magnitude span / 8 자릿수 범위')
ax.set_title('Solar photospheric scale hierarchy / 태양 광구 척도 위계')
plt.tight_layout(); plt.show()

**Insight / 통찰**: The supergranulation scale sits **8 orders of magnitude** above the viscous dissipation scale and 30× above the granulation injection scale. No microscopic-scale argument predicts $L_{\rm SG}$ — this *is* the supergranulation puzzle.

초과립 스케일은 점성 산란 척도보다 **8자릿수**, 과립 주입 척도보다 30배 위에 위치. 미시 척도로는 $L_{\rm SG}$를 예측할 수 없다 — 이것이 초과립 수수께끼.

## Part 2: Synthetic supergranulation velocity field / 합성 초과립 유동

We generate a 2D divergent (incompressible-in-2D-via-velocity-potential) flow whose horizontal velocity has a spectral peak near $\lambda_{\rm SG} = 36$ Mm.

Method: synthesize Fourier modes with amplitude $|\hat\phi(k)| \propto k^{-1}\exp[-(\ln(k/k_{\rm SG}))^2/(2\sigma^2)]$, random phases. Then $\vec v_h = \nabla\phi$ for an outflow-from-cell-centre pattern (matches SG topology).

Fourier 모드를 합성하여 36 Mm 부근에 피크가 있는 발산 유동 생성. 셀 중앙에서 바깥으로 흐르는 SG 토폴로지를 재현.

In [ ]:
def synth_supergranulation(
    nx: int, L_box_Mm: float, lambda_sg_Mm: float,
    sigma_log: float, V_target: float, rng: np.random.Generator,
) -> tuple:
    """Generate a 2D divergent velocity field with a supergranulation-like spectral peak.

    Args:
        nx: Number of grid points per side.
        L_box_Mm: Box side length in Mm.
        lambda_sg_Mm: Target supergranulation wavelength in Mm.
        sigma_log: Log-Gaussian width of the spectral peak.
        V_target: Target rms horizontal velocity in m/s.
        rng: NumPy random generator.

    Returns:
        Tuple (vx, vy, X, Y) with velocities in m/s and coords in Mm.
    """
    dx = L_box_Mm / nx
    coords = np.arange(nx) * dx
    X, Y = np.meshgrid(coords, coords, indexing='xy')

    kx = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
    KX, KY = np.meshgrid(kx, ky, indexing='xy')
    K = np.sqrt(KX**2 + KY**2)
    K_safe = np.where(K == 0, 1, K)

    k_sg = 2 * np.pi / lambda_sg_Mm
    log_ratio = np.log(K_safe / k_sg)
    amp = (1.0 / K_safe) * np.exp(-(log_ratio**2) / (2 * sigma_log**2))
    amp[K == 0] = 0.0

    phases = rng.uniform(0, 2 * np.pi, size=K.shape)
    phi_hat = amp * np.exp(1j * phases)
    phi = np.real(np.fft.ifft2(phi_hat))

    vy_grid, vx_grid = np.gradient(phi, dx)
    rms = np.sqrt(np.mean(vx_grid**2 + vy_grid**2))
    scale = V_target / rms
    return vx_grid * scale, vy_grid * scale, X, Y


nx = 256
L_box = 250.0
vx, vy, X, Y = synth_supergranulation(
    nx=nx, L_box_Mm=L_box, lambda_sg_Mm=36.0,
    sigma_log=0.4, V_target=350.0, rng=rng,
)
speed = np.sqrt(vx**2 + vy**2)
print(f"Box: {L_box} Mm × {L_box} Mm,  grid: {nx}×{nx} (dx={L_box/nx:.2f} Mm)")
print(f"RMS velocity: {np.sqrt(np.mean(vx**2 + vy**2)):.1f} m/s (target 350)")
print(f"Max  velocity: {speed.max():.1f} m/s")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(speed, origin='lower', extent=[0, L_box, 0, L_box], cmap='viridis')
step = 6
ax.quiver(X[::step, ::step], Y[::step, ::step],
          vx[::step, ::step], vy[::step, ::step],
          color='white', alpha=0.6, scale=1.5e4)
ax.set_xlabel('x [Mm]'); ax.set_ylabel('y [Mm]')
ax.set_title('Synthetic supergranulation flow / 합성 초과립 유동\n'
             '(arrows = velocity; colour = speed [m/s])')
plt.colorbar(im, ax=ax, label='|v| [m/s]', fraction=0.045)
plt.tight_layout(); plt.show()

Visually one can spot the supergranule-like cellular structure with diameter ~30 Mm.

셀 직경 ~30 Mm의 초과립 같은 세포 구조 확인 가능.

## Part 3: Horizontal kinetic energy spectrum $E_h(k)$ / 수평 운동에너지 스펙트럼

Reproduce Fig. 4 of the review:

$$
\frac{1}{2}\langle v_h^2\rangle = \int_0^\infty E_h(k)\,dk
$$

Compute the azimuthally-averaged 1D power spectrum and verify the SG peak near $\lambda = 36$ Mm.

방위 평균된 1D 파워 스펙트럼을 계산하여 36 Mm 부근의 SG 피크를 확인.

In [ ]:
def kinetic_energy_spectrum(
    vx: np.ndarray, vy: np.ndarray, L_box_Mm: float, n_bins: int = 60,
) -> tuple:
    """Compute the 1D azimuthally averaged horizontal kinetic energy spectrum.

    Args:
        vx, vy: Velocity components on a square grid [m/s].
        L_box_Mm: Box side length [Mm].
        n_bins: Number of radial wavenumber bins.

    Returns:
        (k_centers [Mm^-1], E_h [m^2/s^2 per Mm^-1]).
    """
    nx = vx.shape[0]
    dx = L_box_Mm / nx
    kx = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
    KX, KY = np.meshgrid(kx, ky, indexing='xy')
    K = np.sqrt(KX**2 + KY**2)

    Vx_hat = np.fft.fft2(vx) / nx**2
    Vy_hat = np.fft.fft2(vy) / nx**2
    P2d = 0.5 * (np.abs(Vx_hat)**2 + np.abs(Vy_hat)**2)

    k_max = K.max()
    edges = np.linspace(0, k_max, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    E = np.zeros(n_bins)
    for i in range(n_bins):
        mask = (K >= edges[i]) & (K < edges[i + 1])
        if mask.any():
            dk = edges[i + 1] - edges[i]
            E[i] = P2d[mask].sum() * (2 * np.pi * centers[i]) / max(mask.sum(), 1) / dk
    return centers, E


k, E_h = kinetic_energy_spectrum(vx, vy, L_box_Mm=L_box)
lam = 2 * np.pi / np.where(k > 0, k, 1)
valid = (k > 0) & (lam < L_box) & (lam > 2 * L_box / nx)
k_peak = k[valid][np.argmax(E_h[valid])]
lambda_peak = 2 * np.pi / k_peak
print(f"Spectral peak at lambda = {lambda_peak:.1f} Mm  (target: 36 Mm)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.loglog(k[valid], E_h[valid], 'C0-', lw=2, label='$E_h(k)$ synthetic / 합성')
ax.axvline(2 * np.pi / 36.0, color='red', ls='--', alpha=0.7,
           label=r'Observed SG peak $\lambda=36$ Mm')
ax.axvline(2 * np.pi / 1.0, color='orange', ls=':', alpha=0.7,
           label=r'Granulation $\lambda=1$ Mm (out of grid range)')
ax2 = ax.secondary_xaxis('top', functions=(lambda x: 2 * np.pi / x, lambda x: 2 * np.pi / x))
ax2.set_xlabel(r'wavelength $\lambda$ [Mm]')
ax.set_xlabel(r'wavenumber $k$ [Mm$^{-1}$]')
ax.set_ylabel(r'$E_h(k)$ [arbitrary]')
ax.set_title('Horizontal kinetic energy spectrum (cf. Hathaway+2000 Fig. 4a)\n'
             '수평 운동에너지 스펙트럼')
ax.legend(loc='lower left'); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

**Comparison with paper / 논문과의 비교**: Hathaway et al. (2000) MDI data shows a clear peak at spherical harmonic $\ell \approx 120$, corresponding to $\lambda \approx 36$ Mm on the solar surface. Our synthetic spectrum reproduces a peak at this wavelength.

Hathaway et al. (2000) MDI 데이터는 구면 조화 $\ell \approx 120$($\lambda \approx 36$ Mm)에서 명확한 피크를 보인다. 합성 스펙트럼이 이 파장에서 피크를 재현.

## Part 4: Anticyclonic signature / 반사이클론 시그너처

Reproduce Fig. 5 of the review (Gizon & Duvall 2003): the correlation between horizontal divergence $D = \nabla \cdot \vec v_h$ and vertical vorticity $\zeta = (\nabla \times \vec v_h)_z$ changes sign across the equator.

We embed our synthetic SG flow on a rotating sphere; Coriolis force $\vec f_C = -2\vec\Omega \times \vec v$ generates vorticity opposite to divergence on outflow cells (anticyclones).

회전 구면에 SG 유동을 두면 Coriolis 힘이 발산-와도 음의 상관(반사이클론)을 만든다. 적도 횡단 시 부호 반전.

In [ ]:
def divergence_vorticity(vx: np.ndarray, vy: np.ndarray, dx_m: float) -> tuple:
    """Compute horizontal divergence and vertical vorticity on a regular grid.

    Args:
        vx, vy: Velocity components [m/s].
        dx_m: Grid spacing [m].

    Returns:
        (divergence [1/s], vertical vorticity [1/s]).
    """
    dvx_dy, dvx_dx = np.gradient(vx, dx_m)
    dvy_dy, dvy_dx = np.gradient(vy, dx_m)
    div = dvx_dx + dvy_dy
    vort = dvy_dx - dvx_dy
    return div, vort


Omega_sun = 2.7e-6  # angular velocity [rad/s]
dx_m = (L_box / nx) * 1e6

latitudes_deg = np.arange(-50, 51, 5)
correlations = []
div0, _ = divergence_vorticity(vx, vy, dx_m)

for lat_deg in latitudes_deg:
    lat = np.deg2rad(lat_deg)
    f_z = 2 * Omega_sun * np.sin(lat)
    coriolis_vorticity = -f_z * div0 * 1e5
    rng_field = np.random.default_rng(seed=int(lat_deg) + 1000)
    vort_field = coriolis_vorticity + 0.5 * np.std(coriolis_vorticity) * rng_field.standard_normal(div0.shape)
    corr = np.corrcoef(div0.ravel(), vort_field.ravel())[0, 1]
    correlations.append(corr)

correlations = np.array(correlations)
print(f"Correlation at lat=+30°: {correlations[latitudes_deg == 30][0]:+.3f}")
print(f"Correlation at lat=  0°: {correlations[latitudes_deg ==  0][0]:+.3f}")
print(f"Correlation at lat=-30°: {correlations[latitudes_deg == -30][0]:+.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(latitudes_deg, correlations, 'o-', lw=2, color='C2')
ax.axhline(0, color='k', lw=0.7, ls=':')
ax.axvline(0, color='k', lw=0.7, ls=':')
ax.set_xlabel(r'Latitude $\lambda$ [deg]')
ax.set_ylabel(r'Correlation(curl, div)')
ax.set_title('Anticyclonic signature of supergranules (cf. Gizon & Duvall 2003 / Fig. 5a)\n'
             '초과립 반사이클론 시그너처')
ax.text(30, 0.5, 'N: negative\n→ anticyclone\n→ outflow + clockwise', ha='center')
ax.text(-30, -0.5, 'S: positive\n→ anticyclone\n→ outflow + counter-CW', ha='center')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**Physics / 물리**: Outflow cells (positive divergence) on a rotating body acquire vorticity *opposite* to the local Coriolis parameter $f = 2\Omega\sin\lambda$. In the northern hemisphere $f > 0$, so anticyclones have $\zeta < 0$; in the south $f < 0$, so $\zeta > 0$. This is identical to the physics of high-pressure systems in Earth's atmosphere.

회전체 위 유출 세포(양의 발산)는 국소 Coriolis 매개변수와 *반대* 부호의 와도를 얻는다. 북반구 $f > 0$이므로 반사이클론은 $\zeta < 0$, 남반구는 그 반대. 지구 대기 고기압계와 동일한 물리.

## Part 5: Magnetic network formation by passive advection / 자기 네트워크 형성

Reproduce Fig. 7 of the review (Roudier et al. 2009): pure passive advection of "corks" by the SG flow concentrates them on cell *boundaries* — the magnetic network pattern.

Integrate $\dot{\vec x}_i = \vec v_h(\vec x_i, t)$ for $N_p$ tracer particles initially distributed uniformly.

SG 유동에 의해 코르크(추적자)가 수동 이류되면 셀 *경계*에 모인다 — 자기 네트워크 패턴.

In [ ]:
def bilinear_velocity(
    x: np.ndarray, y: np.ndarray, vx: np.ndarray, vy: np.ndarray, L_box_Mm: float,
) -> tuple:
    """Bilinear interpolation of velocity at scattered points (periodic).

    Args:
        x, y: Particle positions [Mm].
        vx, vy: Velocity grids [m/s].
        L_box_Mm: Box side length [Mm].

    Returns:
        (u, v) interpolated velocities at particle positions [m/s].
    """
    nx_local = vx.shape[0]
    dx = L_box_Mm / nx_local
    xs = (x % L_box_Mm) / dx
    ys = (y % L_box_Mm) / dx
    i0 = np.floor(xs).astype(int) % nx_local
    j0 = np.floor(ys).astype(int) % nx_local
    i1 = (i0 + 1) % nx_local
    j1 = (j0 + 1) % nx_local
    fx = xs - np.floor(xs)
    fy = ys - np.floor(ys)
    u = ((1 - fx) * (1 - fy) * vx[j0, i0] + fx * (1 - fy) * vx[j0, i1] +
         (1 - fx) * fy * vx[j1, i0] + fx * fy * vx[j1, i1])
    v = ((1 - fx) * (1 - fy) * vy[j0, i0] + fx * (1 - fy) * vy[j0, i1] +
         (1 - fx) * fy * vy[j1, i0] + fx * fy * vy[j1, i1])
    return u, v


def advect_corks(
    vx: np.ndarray, vy: np.ndarray, L_box_Mm: float,
    n_particles: int, dt_s: float, n_steps: int, rng: np.random.Generator,
) -> np.ndarray:
    """Advect passive tracer particles in a steady horizontal velocity field.

    Args:
        vx, vy: Velocity grids [m/s].
        L_box_Mm: Box side length [Mm].
        n_particles: Number of tracer particles.
        dt_s: Time step [s].
        n_steps: Number of integration steps.
        rng: NumPy random generator.

    Returns:
        Array of shape (n_particles, 2) of final positions [Mm].
    """
    pos = rng.uniform(0, L_box_Mm, size=(n_particles, 2))
    Mm_per_m = 1e-6
    for _ in range(n_steps):
        u, v = bilinear_velocity(pos[:, 0], pos[:, 1], vx, vy, L_box_Mm)
        pos[:, 0] = (pos[:, 0] + u * dt_s * Mm_per_m) % L_box_Mm
        pos[:, 1] = (pos[:, 1] + v * dt_s * Mm_per_m) % L_box_Mm
    return pos


n_particles = 6000
dt_s = 600.0  # 10 minutes
n_steps = 12 * 6  # 12 hours of integration

pos_initial = rng.uniform(0, L_box, size=(n_particles, 2))
pos_final = advect_corks(vx, vy, L_box, n_particles, dt_s, n_steps, rng)
print(f"Advected {n_particles} corks for {n_steps*dt_s/3600:.1f} hours.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, sharey=True)
axes[0].scatter(pos_initial[:, 0], pos_initial[:, 1], s=2, color='black', alpha=0.5)
axes[0].set_title('t = 0 — uniform random / 균등 분포')
axes[1].imshow(np.sqrt(vx**2 + vy**2), origin='lower',
               extent=[0, L_box, 0, L_box], cmap='Greys', alpha=0.4)
axes[1].scatter(pos_final[:, 0], pos_final[:, 1], s=2, color='black', alpha=0.7)
axes[1].set_title(f't = {n_steps*dt_s/3600:.0f} h — concentrated on SG boundaries / 경계 집중')
for ax in axes:
    ax.set_xlabel('x [Mm]'); ax.set_ylabel('y [Mm]')
    ax.set_xlim(0, L_box); ax.set_ylim(0, L_box)
fig.suptitle('Magnetic network formation by passive advection (cf. Roudier+2009 / Fig. 7)', y=1.02)
plt.tight_layout(); plt.show()

**Insight / 통찰**: Even purely *kinematic* advection (no MHD feedback) reproduces the network pattern. The review notes (§8.1.2) that this kinematic picture is probably *too simple* — magnetic feedback likely shapes the SG flow itself, making network and SG "a coupled, indistinguishable dynamical product."

순수 *운동학적* 이류만으로도 네트워크 패턴을 재현. 그러나 리뷰는 §8.1.2에서 자기 피드백이 SG 유동 자체를 형성할 가능성을 지적 — 네트워크와 SG는 "결합된 분리 불가능한 동역학적 산물".

## Part 6: Kinetic vs magnetic energy crossover / 운동-자기 에너지 교차

Reproduce Eqs. 7–8 and Fig. 15 of the review:

$$
E_{\rm kin} = 45\left(\frac{\rho}{10^{-3}\,\rm kg/m^3}\right)\left(\frac{V}{300\,\rm m/s}\right)^2\,\rm J/m^3
$$

$$
E_{\rm mag} = 40\left(\frac{B}{100\,\rm G}\right)^2\,\rm J/m^3
$$

Equipartition at $B \sim 100$ G, $V \sim 300$ m/s. kG network elements $\Rightarrow$ super-equipartition by $\sim 100\times$.

100 G, 300 m/s에서 등분배. kG 네트워크 요소는 100배 super-equipartition.

In [ ]:
def kinetic_energy_density(rho_kgm3: float, V_ms: float) -> float:
    """Kinetic energy density [J/m^3]."""
    return 0.5 * rho_kgm3 * V_ms**2


def magnetic_energy_density(B_gauss: float) -> float:
    """Magnetic energy density [J/m^3] from B in Gauss (1 G = 1e-4 T)."""
    mu0 = 4 * np.pi * 1e-7
    B_T = B_gauss * 1e-4
    return B_T**2 / (2 * mu0)


rho = 1e-3  # photospheric density
V_range = np.array([100, 300, 500, 1000])  # m/s
B_range = np.array([10, 60, 100, 130, 1000])  # G

print("Kinetic energy density / 운동에너지 밀도 (rho = 1e-3 kg/m^3):")
for V in V_range:
    print(f"  V={V:5d} m/s : E_kin = {kinetic_energy_density(rho, V):8.2f} J/m^3")

print("\nMagnetic energy density / 자기에너지 밀도:")
for B in B_range:
    print(f"  B={B:5d} G   : E_mag = {magnetic_energy_density(B):8.2f} J/m^3")

V_sg, B_internetwork, B_network = 300, 60, 1000
print(f"\nSG flow:           E_kin({V_sg} m/s) = {kinetic_energy_density(rho, V_sg):.1f} J/m^3")
print(f"Internetwork B:    E_mag({B_internetwork} G)  = {magnetic_energy_density(B_internetwork):.1f} J/m^3")
print(f"Network elements:  E_mag({B_network} G) = {magnetic_energy_density(B_network):.1f} J/m^3")
ratio = magnetic_energy_density(B_network) / kinetic_energy_density(rho, V_sg)
print(f"\nNetwork/SG ratio = {ratio:.0f}  ⇒  super-equipartition by ~{ratio:.0f}×")

In [ ]:
B_grid = np.logspace(0, 4, 200)
V_grid = np.logspace(1, 4, 200)
E_mag_curve = np.array([magnetic_energy_density(B) for B in B_grid])
E_kin_curve = np.array([kinetic_energy_density(rho, V) for V in V_grid])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].loglog(B_grid, E_mag_curve, 'C3-', lw=2)
axes[0].axhline(kinetic_energy_density(rho, V_sg), color='C0', ls='--',
                label=f'$E_{{\\rm kin}}$(SG, V=300 m/s) = 45 J/m³')
axes[0].axvline(100, color='gray', ls=':', label='B = 100 G (equipartition)')
axes[0].axvline(1000, color='black', ls=':', label='Network (kG)')
axes[0].set_xlabel('B [G]'); axes[0].set_ylabel(r'$E_{\rm mag}$ [J/m³]')
axes[0].set_title('Magnetic energy vs field strength')
axes[0].legend(); axes[0].grid(True, which='both', alpha=0.3)

axes[1].loglog(V_grid, E_kin_curve, 'C0-', lw=2)
axes[1].axhline(magnetic_energy_density(100), color='C3', ls='--',
                label='$E_{\\rm mag}$(B=100 G) = 40 J/m³')
axes[1].axvline(300, color='gray', ls=':', label='V = 300 m/s (SG typical)')
axes[1].axvline(1000, color='black', ls=':', label='V = 1 km/s (granulation)')
axes[1].set_xlabel('V [m/s]'); axes[1].set_ylabel(r'$E_{\rm kin}$ [J/m³]')
axes[1].set_title('Kinetic energy vs flow speed (rho=1e-3 kg/m³)')
axes[1].legend(); axes[1].grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

**Insight / 통찰**: At supergranulation parameters (V=300 m/s, B~100 G internetwork) kinetic and magnetic energy densities are comparable (~40-45 J/m³). But network elements with B~1 kG carry ~100× more magnetic energy than the local SG flow can supply — implying the network is *not* simply driven by SG motions but by stronger localized processes (granular advection + flux concentration / partial evacuation).

초과립 매개변수(V=300 m/s, B~100 G)에서 운동·자기 에너지 밀도가 비슷(~40-45 J/m³). 그러나 kG 네트워크 요소는 SG 유동이 공급할 수 있는 양의 ~100배 자기 에너지를 가짐 — 네트워크가 SG에 의해 직접 구동되지 않고 더 강한 국소 과정(과립 이류 + 부분 밀도 비움)에 의해 유지됨을 의미.

## Summary / 요약

| Concept / 개념 | This implementation / 본 구현 | Paper reference / 논문 참조 | Modern equivalent / 현대 동등물 |
|---|---|---|---|
| Dissipation hierarchy / 산란 위계 | Kolmogorov scaling for solar values | §2.2 | DNS direct measurement |
| Synthetic SG flow / 합성 SG 유동 | Fourier-mode synthesis with log-Gaussian peak | Fig. 4, §4.2 | Stein-Nordlund / MURaM realistic MHD sims |
| Power spectrum $E_h(k)$ | FFT + azimuthal average | Eq. 1, Fig. 4 | Hathaway+2000 (MDI), Rieutord+2008 (granule tracking) |
| Anticyclonic vorticity / 반사이클론 와도 | Coriolis acting on synthetic divergence | Fig. 5, §4.5 | Gizon-Duvall 2003 (helioseismology) |
| Network formation / 네트워크 형성 | Passive cork advection (RK1 Euler) | Fig. 7, §4.6.1 | Roudier+2009; Krijger-Roudier 2003 |
| Kin-mag energy ratio / 운동-자기 에너지비 | Closed-form Eqs. 7-8 evaluation | Fig. 15, §8.2.4 | Lites+2008 Hinode polarimetry |

### Connections to other papers / 다른 논문과의 연결

- **Paper #16 (Nordlund+2009, granulation)**: provides the small-scale ($L_G \sim 1$ Mm) injection physics that drives everything in this notebook from below. / 본 노트북의 모든 합성 유동을 아래에서 구동하는 소규모 ($L_G \sim 1$ Mm) 주입 물리.
- **Future LRSP papers on dynamo**: the magnetic feedback story sketched in Part 6 is the bridge to global dynamo models. / Part 6의 자기 피드백은 전역 다이나모 모델로 가는 다리.
- **DKIST + Solar Orbiter (2020s)**: will provide the missing magnetic power spectrum from 100 m to 100 Mm called for in §8.3. / §8.3에서 요구한 100 m–100 Mm 자기력 스펙트럼을 제공할 것.

### Open puzzle reminder / 미해결 수수께끼 환기

The notebook can *generate* a 36 Mm peak by hand-tuning the Fourier amplitudes — but no first-principles simulation of solar convection has yet *spontaneously* produced this peak. That gap is the supergranulation puzzle.

본 노트북은 Fourier 진폭을 손으로 조정해 36 Mm 피크를 *생성*할 수 있지만, 어떤 제일원리(first-principles) 태양 대류 시뮬레이션도 이 피크를 *자발적으로* 만들어내지 못했다. 이 간극이 곧 초과립 수수께끼이다.